# 05 PSO Hyperparameter Tuning (All Datasets)

This notebook runs the clean evaluation pipeline for **all three datasets** (China, COCOMO-81, Desharnais).

**Key features:**
- Leakage-free `X_main` / `X_holdout` split before any scaler fit, PSO tuning, or CV
- Fold-local `StandardScaler` fits inside every CV split
- Baseline and PSO-tuned CNN + MLP models
- EarlyStopping + ReduceLROnPlateau callbacks with internal validation split
- CNN+PSO and MLP+PSO ensemble holdout evaluation
- Standardized metric tables and saved models under `results/metrics/` and `models/`

In [1]:
from importlib import import_module
import importlib.util
import random
from pathlib import Path
import subprocess
import sys
import warnings

import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

warnings.filterwarnings("ignore")

drive = None
if importlib.util.find_spec("google.colab") is not None:
    drive = import_module("google.colab.drive")
    IN_COLAB = True
else:
    IN_COLAB = False


def resolve_project_root() -> Path:
    if IN_COLAB:
        drive.mount("/content/drive", force_remount=False)
        drive_root = Path("/content/drive/MyDrive")
        candidate_roots = [
            drive_root / "Software-cost-Estimation",
            drive_root / "Colab Notebooks" / "Software-cost-Estimation",
        ]
        for candidate in candidate_roots:
            if (candidate / "src").exists():
                return candidate

        for src_dir in drive_root.rglob("src"):
            if src_dir.is_dir() and src_dir.parent.name == "Software-cost-Estimation":
                return src_dir.parent

        raise FileNotFoundError(
            "Could not find the 'Software-cost-Estimation' project folder in Google Drive. "
            "Upload the full project folder to MyDrive or Colab Notebooks."
        )

    root_dir = Path.cwd()
    if not (root_dir / "src").exists() and (root_dir.parent / "src").exists():
        root_dir = root_dir.parent
    return root_dir


root_dir = resolve_project_root()
if not (root_dir / "src").exists():
    raise FileNotFoundError(f"Could not find project root containing 'src' directory. Checked: {root_dir}")

if IN_COLAB and importlib.util.find_spec("pyswarms") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyswarms>=1.3"])

sys.path.insert(0, str(root_dir))
print("Using project root:", root_dir)

from src.cv_pipeline import DATASET_DISPLAY_NAMES, save_benchmark_artifacts
from src.data_loader import load_all_raw_datasets

metrics_dir = root_dir / "results" / "metrics"
models_dir = root_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

raw_datasets = load_all_raw_datasets()
for dataset_key in ("china", "cocomo81", "desharnais"):
    print(f"Loaded {DATASET_DISPLAY_NAMES[dataset_key]}: {raw_datasets[dataset_key].shape}")

print(f"\n{len(raw_datasets)} raw datasets ready for clean holdout + CV benchmarking")

Using project root: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation
Loaded China: (499, 19)
Loaded COCOMO81: (63, 19)
Loaded Desharnais: (81, 13)

3 raw datasets ready for clean holdout + CV benchmarking


In [2]:
TRAINING_EPOCHS = 10
TUNING_EPOCHS = 5
N_PARTICLES = 5
PSO_ITERS = 5

print("Benchmark configuration")
print("- training_epochs:", TRAINING_EPOCHS)
print("- tuning_epochs:", TUNING_EPOCHS)
print("- n_particles:", N_PARTICLES)
print("- pso_iters:", PSO_ITERS)

Benchmark configuration
- training_epochs: 10
- tuning_epochs: 5
- n_particles: 5
- pso_iters: 5


In [3]:
from src.cv_pipeline import run_full_benchmark

print("Running clean holdout + CV benchmark across all datasets...\n")
benchmark_results = run_full_benchmark(
    raw_datasets=raw_datasets,
    use_log_transform=True,
    training_epochs=TRAINING_EPOCHS,
    tuning_epochs=TUNING_EPOCHS,
    n_particles=N_PARTICLES,
    iters=PSO_ITERS,
    verbose=0,
)

holdout_results = benchmark_results["holdout_results"]
full_comparison_final = benchmark_results["full_comparison_final"]

print("All datasets complete.")

Running clean holdout + CV benchmark across all datasets...




2026-04-29 16:07:43,067 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|5/5, best_cost=1.09e+6
2026-04-29 16:09:04,107 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 1085107.468278216, best pos: [ 8.76555395  3.69919383 -2.00575867  0.15051307  2.93495458 10.68711251]
2026-04-29 16:22:35,033 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|5/5, best_cost=8.01e+3
2026-04-29 16:23:34,199 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 8013.485026991197, best pos: [ -3.39824388   0.27772476   1.39440422 215.1648427   45.59290098]
2026-04-29 16:28:37,681 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|5/5, best_cost=324
2026-04-29 16:29:17,139 - pyswarms.sing

All datasets complete.


In [4]:
print("Final holdout results")
print("=" * 70)
display(holdout_results)

print("\n5-fold cross-validation final comparison on X_main")
print("=" * 70)
full_comparison_final

Final holdout results


,Dataset,Model,RMSE,MAE,R2,MAPE,MdMRE,Pred25
0,China,LinearRegression,53165.342106,7281.103990,-93.037921,126.831854,0.483402,18.000000
1,China,RandomForest,1634.160681,392.842685,0.911155,10.733604,0.077702,98.000000
2,China,XGBoost,1467.321352,367.765243,0.928370,9.590352,0.062659,94.000000
3,China,CNN_baseline,98334.093398,16671.253203,-320.702199,610.360125,0.889825,12.000000
4,China,CNN_PSO,3463.521400,1468.666637,0.600900,45.474434,0.376029,33.000000
5,China,MLP_baseline,9308.433189,3753.675303,-1.882697,126.341274,0.993758,0.000000
6,China,MLP_PSO,6349.519403,3167.623193,-0.341305,102.448529,0.996744,0.000000
7,China,CNN_PSO_ensemble,2381.084604,916.908206,0.811377,45.779683,0.250699,50.000000
8,China,MLP_PSO_ensemble,757433.805280,79197.816601,-19085.889629,3258.896291,0.971308,1.000000
9,COCOMO81,LinearRegression,395.080548,240.920933,0.494898,262.738931,0.588853,15.384615



5-fold cross-validation final comparison on X_main


,Dataset,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,R2_mean,R2_std,MAPE_mean,MAPE_std,MdMRE_mean,MdMRE_std,Pred25_mean,Pred25_std
0,China,LinearRegression,4.823992e+06,8.900180e+06,5.431055e+05,9.950693e+05,-2.212662e+06,4.413961e+06,2.173859e+03,3.964524e+03,0.433212,0.014121,26.072785,2.630876
1,China,RandomForest,1.285643e+03,6.281309e+02,4.481264e+02,1.377663e+02,9.618910e-01,2.220509e-02,1.071370e+01,1.215741e+00,0.080660,0.009387,96.243671,2.232563
2,China,XGBoost,1.549659e+03,8.961370e+02,4.530064e+02,1.387175e+02,9.433224e-01,4.300237e-02,9.024719e+00,1.228912e+00,0.061628,0.006453,95.740506,1.692887
3,China,CNN_baseline,1.646439e+10,3.070348e+10,1.840779e+09,3.432754e+09,-1.580569e+13,3.139610e+13,3.599949e+06,6.206174e+06,0.978393,0.001931,0.250000,0.500000
4,China,CNN_PSO,5.560666e+05,7.731438e+05,6.332464e+04,8.646446e+04,-2.282090e+04,4.007162e+04,6.012121e+02,9.561916e+02,0.294508,0.033188,43.335443,4.876555
5,China,MLP_baseline,8.430783e+08,1.205452e+09,9.426262e+07,1.347742e+08,-3.129405e+10,4.813031e+10,4.024393e+05,5.457728e+05,0.997345,0.000385,0.000000,0.000000
6,China,MLP_PSO,1.395784e+09,2.748855e+09,1.560570e+08,3.073313e+08,-2.437241e+11,4.874005e+11,1.698515e+06,3.388049e+06,0.998541,0.000226,0.000000,0.000000
7,China,CNN_PSO_ensemble,7.780540e+04,9.495854e+04,9.524815e+03,1.088848e+04,-3.244130e+02,5.651788e+02,7.232967e+01,5.294069e+01,0.201378,0.030115,59.164557,5.346307
8,China,MLP_PSO_ensemble,2.932063e+33,5.864125e+33,3.278146e+32,6.556291e+32,-1.102358e+60,2.204717e+60,3.604735e+30,7.209469e+30,0.929616,0.090628,3.253165,4.648912
9,COCOMO81,LinearRegression,6.603249e+05,1.318321e+06,2.089929e+05,4.169795e+05,-1.622856e+05,3.245604e+05,3.305590e+03,6.299014e+03,0.865062,0.496419,22.000000,7.483315


In [5]:
saved_paths = save_benchmark_artifacts(
    benchmark_results=benchmark_results,
    metrics_dir=metrics_dir,
    models_dir=models_dir,
)

for label, path in saved_paths.items():
    print(f"Saved {label}: {path}")

2026-04-29 21:41:35,939 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 


2026-04-29 21:41:36,021 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
2026-04-29 21:41:36,069 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
2026-04-29 21:41:36,094 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
2026-04-29 21:41:36,123 - ab

Saved holdout_results: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\results\metrics\holdout_results.csv
Saved full_comparison_final: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\results\metrics\full_comparison_final.csv
Saved cnn_vs_pso_metrics: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\results\metrics\cnn_vs_pso_metrics.csv
Saved best_hyperparams: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models\best_hyperparams.json
Saved training_histories: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models\training_histories.json
Saved cnn_baseline_china: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models\cnn_baseline_china.h5
Saved cnn_pso_china: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models\cnn_pso_china.h5
Saved mlp_baseline_china: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models\mlp_baseline_china.h5
Saved mlp_pso_china: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\models